# 01 — Data Exploration

Explore the KINNEWS/KIRNEWS news classification corpus and AfriSenti sentiment dataset.
We examine class distributions, text length statistics, and vocabulary overlap between Kinyarwanda and Kirundi.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

## 1. KINNEWS / KIRNEWS Dataset

In [ ]:
# Load preprocessed data
kin_train = pd.read_csv('../data/processed/kin_train.csv')
kin_test  = pd.read_csv('../data/processed/kin_test.csv')
kir_train = pd.read_csv('../data/processed/kir_train.csv')
kir_test  = pd.read_csv('../data/processed/kir_test.csv')

print('Kinyarwanda: train', len(kin_train), '| test', len(kin_test))
print('Kirundi:     train', len(kir_train), '| test', len(kir_test))

In [ ]:
CATEGORY_NAMES = {
    0: 'Politics', 1: 'Sport', 2: 'Economy', 3: 'Health',
    4: 'Entertainment', 5: 'History', 6: 'Technology',
    7: 'Culture', 8: 'Religion', 9: 'Environment',
    10: 'Education', 11: 'Relationship'
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (df, title) in zip(axes, [(kin_train, 'Kinyarwanda Train'), (kir_train, 'Kirundi Train')]):
    counts = df['label'].value_counts().sort_index()
    ax.bar([CATEGORY_NAMES.get(i, str(i)) for i in counts.index], counts.values, color='steelblue')
    ax.set_title(title)
    ax.set_xlabel('Category')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('../paper/figures/dataset_distribution.pdf', bbox_inches='tight')
plt.show()
print('Class imbalance ratio (Kinyarwanda):', kin_train['label'].value_counts().iloc[0] / kin_train['label'].value_counts().iloc[-1])

In [ ]:
# Text length analysis
for name, df in [('Kinyarwanda', kin_train), ('Kirundi', kir_train)]:
    df['text_len'] = df['text'].str.split().str.len()
    print(f'{name}: mean={df["text_len"].mean():.1f} words | median={df["text_len"].median():.0f} | max={df["text_len"].max()}')

## 2. Vocabulary Overlap Between Kinyarwanda and Kirundi

In [ ]:
from collections import Counter

def get_vocab(df):
    words = set()
    for text in df['text'].dropna():
        words.update(str(text).lower().split())
    return words

kin_vocab = get_vocab(kin_train)
kir_vocab = get_vocab(kir_train)
overlap = kin_vocab & kir_vocab

print(f'Kinyarwanda vocab: {len(kin_vocab):,}')
print(f'Kirundi vocab:     {len(kir_vocab):,}')
print(f'Shared tokens:     {len(overlap):,} ({100*len(overlap)/len(kin_vocab|kir_vocab):.1f}% of union)')

## 3. AfriSenti Sentiment Distribution

In [ ]:
import os
sentiment_files = [f for f in os.listdir('../data/processed/afrisenti/') if f.endswith('_train.csv')]

SENTIMENT_LABELS = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
fig, axes = plt.subplots(1, min(3, len(sentiment_files)), figsize=(14, 4))
for ax, fname in zip(axes, sentiment_files[:3]):
    df = pd.read_csv(f'../data/processed/afrisenti/{fname}')
    lang = fname.split('_')[0]
    counts = df['label'].value_counts().sort_index()
    ax.pie(counts.values, labels=[SENTIMENT_LABELS.get(i, str(i)) for i in counts.index], autopct='%1.1f%%')
    ax.set_title(f'AfriSenti — {lang.upper()}')
plt.tight_layout()
plt.show()